# `verl-tool` — test-agent (LLM + tool server)

This notebook demonstrates a **real** multi-turn agent loop:

1) an LLM generates an **action** (a text tool-call)
2) the tool server executes it (`python_code`)
3) we feed the **observation** back to the LLM

This is still **not training** (no PPO). It is an integration test for the interaction loop.


In [11]:
from __future__ import annotations

import os
import sys
from pathlib import Path

HERE = Path.cwd().resolve()
VERL_TOOL_ROOT = HERE
for _ in range(4):
    if (VERL_TOOL_ROOT / "pyproject.toml").exists() and (VERL_TOOL_ROOT / "verl_tool").exists():
        break
    VERL_TOOL_ROOT = VERL_TOOL_ROOT.parent

print("python=", sys.executable)
print("VERL_TOOL_ROOT=", VERL_TOOL_ROOT)
assert (VERL_TOOL_ROOT / "verl_tool").exists(), "Could not locate verl-tool package root"


python= /Users/lzanda/Desktop/VRTOOL-Framework/verl-tool/.venv/bin/python3
VERL_TOOL_ROOT= /Users/lzanda/Desktop/VRTOOL-Framework/verl-tool


## Configure the LLM endpoint 
Ollama provides an OpenAI-compatible endpoint, so you can run locally without OpenAI credits:
- `OPENAI_BASE_URL=http://localhost:11434/v1`
- `OPENAI_API_KEY=ollama` (ignored by Ollama)
- `OPENAI_MODEL=llama3.2:3b` (or any model you pulled with `ollama pull`)


In [12]:
# Notebook kernels do NOT automatically inherit `export ...` from your terminal.
# If you want to run here, set env vars in this cell.
#
# IMPORTANT: avoid pasting real keys in the notebook (it can get saved to disk).

import getpass

USE_OLLAMA = True  # set False to use OpenAI hosted

if USE_OLLAMA:
    # Ollama (local) — OpenAI-compatible
    # IMPORTANT: force a local Ollama model id here, even if OPENAI_MODEL was set earlier.
    os.environ["OPENAI_BASE_URL"] = "http://localhost:11434"
    os.environ["OPENAI_API_KEY"] = "ollama"  # ignored by Ollama
    os.environ["OPENAI_MODEL"] = "llama3.2:3b"  # change to any `ollama list` model
else:
    # OpenAI hosted
    os.environ.setdefault("OPENAI_BASE_URL", "https://api.openai.com")
    if not os.getenv("OPENAI_MODEL"):
        os.environ["OPENAI_MODEL"] = "gpt-4.1-mini"  # change if you want
    if not os.getenv("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter OPENAI_API_KEY (input hidden): ")

print("OPENAI_BASE_URL=", os.getenv("OPENAI_BASE_URL"))
print("OPENAI_MODEL=", os.getenv("OPENAI_MODEL"))
print("OPENAI_API_KEY set?", bool(os.getenv("OPENAI_API_KEY")))


OPENAI_BASE_URL= http://localhost:11434
OPENAI_MODEL= llama3.2:3b
OPENAI_API_KEY set? True


In [13]:
# Start the tool server from the notebook (recommended).
# It auto-selects a free port to avoid "address already in use" problems.
#
# Important: if you re-run this cell, we must NOT change TOOL_SERVER_URL while
# keeping an old server process running. So we reuse the previous URL if the
# process is still alive.

import socket
import time
import subprocess


def pick_free_port(host: str = "127.0.0.1") -> int:
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.bind((host, 0))
    port = int(s.getsockname()[1])
    s.close()
    return port


def port_is_open(host: str, port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(0.2)
        return s.connect_ex((host, port)) == 0


HOST = "127.0.0.1"

# Reuse existing server if still running
TOOL_SERVER_PROC = globals().get("TOOL_SERVER_PROC")
existing_url = globals().get("TOOL_SERVER_URL")

if TOOL_SERVER_PROC is not None and TOOL_SERVER_PROC.poll() is None and isinstance(existing_url, str):
    print(f"Tool server already running (pid={TOOL_SERVER_PROC.pid}).")
    TOOL_SERVER_URL = existing_url
else:
    PORT = pick_free_port(HOST)
    TOOL_SERVER_URL = f"http://{HOST}:{PORT}/get_observation"

    cmd = [
        sys.executable,
        "-m",
        "verl_tool.servers.tool_server",
        "--host",
        HOST,
        "--port",
        str(PORT),
        "--tool_type",
        "python_code",
        "--workers_per_tool",
        "2",
        "--http",
        "h11",
    ]
    print("$", " ".join(cmd))
    TOOL_SERVER_PROC = subprocess.Popen(
        cmd,
        cwd=str(VERL_TOOL_ROOT),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    globals()["TOOL_SERVER_PROC"] = TOOL_SERVER_PROC
    globals()["TOOL_SERVER_URL"] = TOOL_SERVER_URL

    # Wait briefly for the port to open
    t0 = time.time()
    while time.time() - t0 < 10.0:
        if port_is_open(HOST, PORT):
            break
        time.sleep(0.2)

print("TOOL_SERVER_URL=", TOOL_SERVER_URL)


Tool server already running (pid=21342).
TOOL_SERVER_URL= http://127.0.0.1:62712/get_observation


## Tool server

Recommended: start it **from this notebook** using the cell above (it picks a free port automatically).

Terminal fallback (if you prefer manual):

```bash
python3 -m verl_tool.servers.tool_server --host 127.0.0.1 --port 5500 --tool_type python_code --workers_per_tool 2 --http h11
```

If port `5500` is busy, change `--port`.


In [14]:
# Change the user task (prompt) here and re-run to compare trajectories.
TASK = "Compute (12345 * 6789) and return the exact integer."

# Examples:
# TASK = "What is the sum of squares from 1 to 100? Use the python tool."
# TASK = "Compute the 50th Fibonacci number exactly. Use python."

print("TASK=", TASK)


TASK= Compute (12345 * 6789) and return the exact integer.


In [15]:
import json
import subprocess
from pathlib import Path

script = VERL_TOOL_ROOT / "scripts" / "test-agent" / "run_agent_loop_openai_compatible.py"
assert script.exists(), f"Missing: {script}"

# Uses TOOL_SERVER_URL set by the previous cell.
dump_path = Path.cwd() / "trajectory_test_agent.json"

cmd = [
    sys.executable,
    str(script),
    "--tool-server-url",
    TOOL_SERVER_URL,
    "--task",
    TASK,
    "--dump-trajectory-json",
    str(dump_path),
]
print("$", " ".join(cmd))

res = subprocess.run(cmd, cwd=str(VERL_TOOL_ROOT), text=True, capture_output=True)
print("exit_code=", res.returncode)
print("--- stdout ---")
print(res.stdout)
print("--- stderr ---")
print(res.stderr)

if res.returncode != 0:
    raise SystemExit("Agent loop test failed. See output above.")

print(f"\nTrajectory JSON written to: {dump_path}")
print("\nPreview:")
traj = json.loads(dump_path.read_text(encoding="utf-8"))
print({k: traj[k] for k in ["trajectory_id", "stop_reason", "task"] if k in traj})


$ /Users/lzanda/Desktop/VRTOOL-Framework/verl-tool/.venv/bin/python3 /Users/lzanda/Desktop/VRTOOL-Framework/verl-tool/scripts/test-agent/run_agent_loop_openai_compatible.py --tool-server-url http://127.0.0.1:62712/get_observation --task Compute (12345 * 6789) and return the exact integer. --dump-trajectory-json /Users/lzanda/Desktop/VRTOOL-Framework/verl-tool/scripts/test-agent/trajectory_test_agent.json
exit_code= 0
--- stdout ---
[INFO] trajectory_id= test-agent-cd9a165ecc
[INFO] tool_server_url= http://127.0.0.1:62712/get_observation
[INFO] openai_compat_base_url= http://localhost:11434
[INFO] model= llama3.2:3b

[INFO] Turn 1: asking LLM for next action/answer...
--- LLM output ---
```python
print(12345 * 6789)
```
------------------

[INFO] tool_response valid= True done= False
--- observation (from tool server) ---

```result
83810205
```

-------------------------------------

[INFO] Turn 2: asking LLM for next action/answer...
--- LLM output ---
The result of the computation (1

In [16]:
# Render the trajectory as a small table
from typing import Any

def as_table(rows: list[dict[str, Any]], cols: list[str]) -> str:
    def cell(v: Any) -> str:
        s = v if isinstance(v, str) else json.dumps(v, ensure_ascii=False)
        s = s.replace("\n", " ")
        if len(s) > 80:
            s = s[:80] + "…"
        return s

    header = "| " + " | ".join(cols) + " |"
    sep = "|" + "|".join(["---"] * (len(cols) + 2))
    body = []
    for r in rows:
        body.append("| " + " | ".join(cell(r.get(c, "")) for c in cols) + " |")
    return "\n".join([header, sep] + body)

turn_rows = []
for t in traj.get("turns", []):
    turn_rows.append(
        {
            "turn": t.get("turn"),
            "llm_latency_ms": round(float(t.get("llm_latency_ms", 0.0)), 1),
            "tool_latency_ms": round(float(t.get("tool_latency_ms", 0.0)), 1),
            "tool_valid": t.get("tool_response_valid"),
            "llm_output_preview": (t.get("llm_output") or "").strip(),
            "observation_preview": t.get("tool_observation"),
        }
    )

md = as_table(
    turn_rows,
    ["turn", "llm_latency_ms", "tool_latency_ms", "tool_valid", "llm_output_preview", "observation_preview"],
)
print(md)


| turn | llm_latency_ms | tool_latency_ms | tool_valid | llm_output_preview | observation_preview |
|---|---|---|---|---|---|---|---
| 1 | 590.0 | 69.6 | true | ```python print(12345 * 6789) ``` |  ```result 83810205 ```  |
